In [ ]:
import pandas as pd
import numpy as np
import os
import time

In [ ]:
current_path = os.getcwd()
path_downloads = 'C:\\Users\\user\\Downloads'

In [ ]:
#Get files for df
def get_last(path):
    os.chdir(path)
    files = os.listdir(os.getcwd())
    time = map(os.path.getmtime,files) 
    sorted_files = sorted(zip(files,time),key=lambda x: x[1],reverse=True)
    os.chdir(current_path)
    return (sorted_files[0][0],sorted_files[1][0])
sale_file,promo_file = get_last(path_downloads)

In [ ]:
#Creating dataframes
if '.csv' in promo_file:
    promo_df = pd.read_csv(path_downloads + '\\' + promo_file,encoding='utf8').drop(0)
else:
    promo_df = pd.read_excel(path_downloads + '\\' + promo_file).drop(0)
sale_df = pd.read_excel(path_downloads + '\\' +sale_file).drop(0)
lpc = pd.read_excel('lpc.xlsx')

In [ ]:
sale_df.drop(['КаналПродаж', 'Агент', 
       'Торговая точка', 'Адрес', 'Документ', 'Дата',
       'КодТовара', 'Продажи вторичные, грн.',],axis=1,inplace=True)
sale_df['Цена'] = sale_df['Цена'].astype(str).str.strip()
sale_df.replace({'Цена':["-",'',"\t"]},"0",inplace=True)
sale_df['Цена'] = sale_df['Цена'].str.replace(',','.')
sale_df['Цена'].fillna("0",inplace=True)
sale_df['Цена'] = sale_df['Цена'].astype(float)
sale_df.drop(sale_df[(sale_df['Цена'] < -1 ) | (sale_df['Цена'] > 1)]["Цена"].index,inplace=True)
sale_df['Цена'].unique()
sale_df.drop('Цена',axis=1,inplace=True)
sale_df.replace({'Группа':['Вино полусладкое','Вино сухое']},'Вино классика',inplace=True)
sale_df.replace({'Группа':['Шампанское 1821']},'Шампанское',inplace=True)
sale_df.rename(columns={'Группа':'Группа текущая','Регион':'Регион текущий','КодТТ':'КодТорговойТочки'},inplace=True)
sale_df.groupby(by=sale_df.columns.to_list()).sum().reset_index()
sale_df.replace({'Регион текущий': {'Белгород': 'Одесса', 'Котовск': 'Одесса','Измаил': 'Одесса','Вознесенск': 'Николаев'}},inplace=True)
sale_df.drop(sale_df[sale_df['Регион текущий']=='Регион не указан'].index,inplace=True)

In [ ]:
#Checking types
promo_df.dtypes

In [ ]:
#Reformat date format and numeric
promo_df = promo_df[promo_df['Регион текущий']!= 'Регион не указан']
sale_df = sale_df[sale_df['Регион текущий']!= 'Регион не указан']
promo_df['Дата'] = pd.to_datetime(promo_df['Дата'],dayfirst=True)
for i in ['Продажи вторичные, бут.', 'Подарок']:
    promo_df[i] = promo_df[i].astype(str).str.split().apply("".join).apply(pd.to_numeric)
promo_df['Продажи вторичные, л.'] = promo_df['Продажи вторичные, л.'].astype(str).str.replace(',','.').str.split().apply("".join).astype(float)

In [ ]:
#Checking types
promo_df.dtypes

In [ ]:
promo_df.head()

In [ ]:
#Split litres measure sales activity into litre groups 
sale_df.loc[(sale_df['SKU'].str.contains('0,25')) & (sale_df['Группа текущая'].str.contains('Bolgrad')),'Группа текущая'] = "Коньяк Bolgrad 0,25"
sale_df.loc[(sale_df['SKU'].str.contains('0,5')) & (sale_df['Группа текущая'].str.contains('Bolgrad')),'Группа текущая'] = "Коньяк Bolgrad 0,5"
sale_df.loc[(sale_df['SKU'].str.contains('0,25')) & (sale_df['Группа текущая'].str.contains('Армения')),'Группа текущая'] = "коньяк Армения 0,25"
sale_df.loc[(sale_df['SKU'].str.contains('0,5')) & (sale_df['Группа текущая'].str.contains('Армения')),'Группа текущая'] = "коньяк Армения 0,5"
sale_df.loc[(sale_df['SKU'].str.contains('0,1')) & (sale_df['Группа текущая'].str.contains('Bolgrad')),'Группа текущая'] = "Коньяк Болградский"

In [ ]:
sale_df['Группа текущая'].unique()

In [ ]:
#Creating list of litres sku and litre groups 
litre_sku = ["Коньяк Bolgrad 0,25","Коньяк Bolgrad 0,5","коньяк Армения 0,25","коньяк Армения 0,5"]
litre_group = ["Коньяк Bolgrad","коньяк Армения"]

In [ ]:
#Сreating table with distributors bottle request
request = sale_df.groupby(by=['Дистрибьютор', 'Регион текущий','Группа текущая']).sum()
request.rename(columns={'Продажи вторичные, бут.':'Дистр'},inplace=True)
request

In [ ]:
#Generating summary of sale and promo in non volume measure groups
sale_nonl_group = sale_df[~sale_df['Группа текущая'].isin(litre_sku)]\
     .groupby(by=['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки', 'Группа текущая']).sum()
promo_nonl_group = promo_df[~promo_df['Группа текущая'].isin(litre_group)]\
     .groupby(by=['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки', 'Группа текущая']).sum()
sale_nonl_group.rename(columns={'Продажи вторичные, бут.':"Выдано"},inplace=True)

In [ ]:
sale_nonl_group

In [ ]:
promo_nonl_group

In [ ]:
# Matching actual and possible issue bottles in non volume measure groups 
result_nonl = sale_nonl_group.join(promo_nonl_group,how='left')
result_nonl = result_nonl.fillna(0)
result_nonl["Разница"] = result_nonl['Выдано'] - result_nonl['Подарок']
result_nonl = result_nonl[result_nonl["Разница"]>0].drop(['Продажи вторичные, бут.','Продажи вторичные, л.','Количество SKU (2х)'],axis=1)
result_nonl.rename(columns={'Подарок':'Подтверждено'},inplace=True)
ressult_in_non_litre  = result_nonl[['Выдано','Подтверждено','Разница']]
ressult_in_non_litre.astype(int)

In [ ]:
#Extract part with group which is participated in volume activitities measure 
sale_l_group = sale_df[sale_df['Группа текущая'].isin(litre_sku)]
promo_l_group = promo_df[promo_df['Группа текущая'].isin(litre_group)]

In [ ]:
#Split into different df by groups
sale_l_group_bg = sale_l_group[sale_l_group['Группа текущая'].str.contains("Коньяк Bolgrad")].pivot_table(index=['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки'], columns='Группа текущая', aggfunc=np.sum).fillna(0)
sale_l_group_ar = sale_l_group[sale_l_group['Группа текущая'].str.contains("коньяк Армения")].pivot_table(index=['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки'], columns='Группа текущая', aggfunc=np.sum).fillna(0)
promo_l_group_bg = promo_l_group[promo_l_group['Группа текущая'].str.contains("Коньяк Bolgrad")].pivot_table(index=['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки'], columns='Группа текущая', values='Подарок', aggfunc=np.sum).fillna(0)
promo_l_group_ar = promo_l_group[promo_l_group['Группа текущая'].str.contains("коньяк Армения")].pivot_table(index=['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки'], columns='Группа текущая', values='Подарок', aggfunc=np.sum).fillna(0)

In [ ]:
#Drop multilevel in columns, evaluate required volume for confirmation issue
sale_l_group_ar.columns = sale_l_group_ar.columns.droplevel(0)
sale_l_group_bg.columns = sale_l_group_bg.columns.droplevel(0)
sale_l_group_ar['litre'] = sale_l_group_ar['коньяк Армения 0,25']*3 + sale_l_group_ar['коньяк Армения 0,5']*5
sale_l_group_bg['litre'] = sale_l_group_bg['Коньяк Bolgrad 0,25'] + sale_l_group_bg['Коньяк Bolgrad 0,5']*2
promo_l_group_bg.rename(columns={promo_l_group_bg.columns[0]:'Подарок'},inplace=True)
promo_l_group_ar.rename(columns={promo_l_group_ar.columns[0]:'Подарок'},inplace=True)

In [ ]:
# Matching actual and possible issue volume, evaluating difference 
result_l_ar = sale_l_group_ar.join(promo_l_group_ar)
result_l_bg = sale_l_group_bg.join(promo_l_group_bg)
result_l_ar["Diff"] = result_l_ar['litre'] - result_l_ar['Подарок']
result_l_bg["Diff"] = result_l_bg['litre'] - result_l_bg['Подарок']
result_l_bg.fillna(0,inplace=True)
result_l_ar.fillna(0,inplace=True)
result_l_bg.astype(int)
result_l_ar.astype(int)

In [ ]:
result_l_bg.reset_index()[result_l_bg.reset_index()['КодТорговойТочки']=='48-99905817'].drop(['litre','Подарок','Diff'],axis=1).fillna(0).stack()
# .append(result_l_ar).drop(['litre','Подарок','Diff'],axis=1).fillna(0)

In [ ]:
#Confirmed bottles in volume measure promos for creating explanation column with string of sales details
confirmed_litre = result_l_bg.append(result_l_ar).drop(['litre','Подарок','Diff'],axis=1).fillna(0).stack()
confirmed_litre = confirmed_litre[confirmed_litre>0]
confirmed_litre.index.names = ['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки', 'Группа текущая 2']
confirmed_litre.name = 'Бут'
confirmed_litre

In [ ]:
#Save only sales where difference more than 0 or where the condition is not met
result_l_bg = result_l_bg[result_l_bg["Diff"] > 0].reset_index()
result_l_ar = result_l_ar[result_l_ar["Diff"] > 0].reset_index()

In [ ]:
#Creating interim column for conversion litre to holding bootles which will be used in next evaluation
result_l_bg["temp"] = result_l_bg['Коньяк Bolgrad 0,25'] - result_l_bg["Подарок"]
result_l_ar["temp"] = result_l_ar['коньяк Армения 0,25']*3 - result_l_ar["Подарок"]

In [ ]:
#Creating and fillin zeroes columns for holding bootles
result_l_bg["Удержание Коньяк Bolgrad 0,25"] = result_l_bg["Удержание Коньяк Bolgrad 0,5"] = 0
result_l_ar["Удержание коньяк Армения 0,25"] = result_l_ar["Удержание коньяк Армения 0,5"] = 0

In [ ]:
#Filling holding values where both sku in shipment
result_l_bg.loc[(result_l_bg['Коньяк Bolgrad 0,25'] != 0) & (result_l_bg['Коньяк Bolgrad 0,5'] != 0) & (result_l_bg['temp']>=0),'Удержание Коньяк Bolgrad 0,5'] =  result_l_bg['Коньяк Bolgrad 0,5']
result_l_bg.loc[(result_l_bg['Коньяк Bolgrad 0,25'] != 0) & (result_l_bg['Коньяк Bolgrad 0,5'] != 0) & (result_l_bg['temp']<0),'Удержание Коньяк Bolgrad 0,5'] = \
                                                                        np.ceil(((result_l_bg['temp']) + result_l_bg['Коньяк Bolgrad 0,5']*2)/2)
result_l_bg.loc[(result_l_bg['Коньяк Bolgrad 0,25'] != 0) & (result_l_bg['Коньяк Bolgrad 0,5'] != 0) & (result_l_bg['temp']>0),'Удержание Коньяк Bolgrad 0,25'] = result_l_bg['temp']
result_l_ar.loc[(result_l_ar['коньяк Армения 0,25'] != 0) & (result_l_ar['коньяк Армения 0,5'] != 0) & (result_l_ar['temp']>=0),'Удержание коньяк Армения 0,5'] = result_l_ar['коньяк Армения 0,5']
result_l_ar.loc[(result_l_ar['коньяк Армения 0,25'] != 0) & (result_l_ar['коньяк Армения 0,5'] != 0) & (result_l_ar['temp']<0),'Удержание коньяк Армения 0,5'] =\
                                                                        np.ceil(((result_l_ar['temp']) + result_l_ar['коньяк Армения 0,5']*5)/5)
result_l_ar.loc[(result_l_ar['коньяк Армения 0,25'] != 0) & (result_l_ar['коньяк Армения 0,5'] != 0) & (result_l_ar['temp']>0),'Удержание коньяк Армения 0,25'] = np.ceil(result_l_ar['temp']/3)

In [ ]:
#Filling holding values where one of two sku's in shipment
result_l_bg.loc[result_l_bg['Коньяк Bolgrad 0,5'] == 0, 'Удержание Коньяк Bolgrad 0,25'] = result_l_bg["Diff"]
result_l_bg.loc[result_l_bg['Коньяк Bolgrad 0,25'] == 0, 'Удержание Коньяк Bolgrad 0,5'] = np.ceil(result_l_bg["Diff"]/2)
result_l_ar.loc[result_l_ar['коньяк Армения 0,5'] == 0, 'Удержание коньяк Армения 0,25'] = np.ceil(result_l_ar["Diff"]/3)
result_l_ar.loc[result_l_ar['коньяк Армения 0,25'] == 0, 'Удержание коньяк Армения 0,5'] = np.ceil(result_l_ar["Diff"]/5)

In [ ]:
# Spliting into different sku's matching
btl_25_ar = result_l_ar.loc[result_l_ar['Удержание коньяк Армения 0,5']==0,~result_l_ar.columns.isin\
                (['litre','Подарок','Diff','temp','коньяк Армения 0,5','Удержание коньяк Армения 0,5'])].rename_axis(None, axis=1)
btl_25_ar['Группа текущая'] = 'коньяк Армения 0,25'
btl_25_ar['Подтверждено'] = btl_25_ar['коньяк Армения 0,25'] - btl_25_ar['Удержание коньяк Армения 0,25']
btl_25_ar.rename(columns={'коньяк Армения 0,25':'Выдано','Удержание коньяк Армения 0,25':'Разница'},inplace=True)
# btl_25_ar.set_index(list(btl_25_ar.drop(['Выдано','Разница','Подтверждено'],axis=1).columns.values))

btl_5_ar = result_l_ar.loc[result_l_ar['Удержание коньяк Армения 0,25']==0,~result_l_ar.columns.isin\
                (['litre','Подарок','Diff','temp','коньяк Армения 0,25','Удержание коньяк Армения 0,25'])].rename_axis(None, axis=1)
btl_5_ar['Группа текущая'] = 'коньяк Армения 0,5'
btl_5_ar['Подтверждено'] = btl_5_ar['коньяк Армения 0,5'] - btl_5_ar['Удержание коньяк Армения 0,5']
btl_5_ar.rename(columns={'коньяк Армения 0,5':'Выдано','Удержание коньяк Армения 0,5':'Разница'},inplace=True)
# btl_5_ar.set_index(list(btl_5_ar.drop(['Выдано','Разница','Подтверждено'],axis=1).columns.values))

btl_25_bg= result_l_bg.loc[result_l_bg['Удержание Коньяк Bolgrad 0,5']==0,~result_l_bg.columns.isin\
                (['litre','Подарок','Diff','temp','Коньяк Bolgrad 0,5','Удержание Коньяк Bolgrad 0,5'])].rename_axis(None, axis=1)
btl_25_bg['Группа текущая'] = 'Коньяк Bolgrad 0,25'
btl_25_bg['Подтверждено'] = btl_25_bg['Коньяк Bolgrad 0,25'] - btl_25_bg['Удержание Коньяк Bolgrad 0,25']
btl_25_bg.rename(columns={'Коньяк Bolgrad 0,25':'Выдано','Удержание Коньяк Bolgrad 0,25':'Разница'},inplace=True)
# btl_25_bg.set_index(list(btl_25_bg.drop(['Выдано','Разница','Подтверждено'],axis=1).columns.values))

btl_5_bg = result_l_bg.loc[result_l_bg['Удержание Коньяк Bolgrad 0,25']==0,~result_l_bg.columns.isin\
                (['litre','Подарок','Diff','temp','Коньяк Bolgrad 0,25','Удержание Коньяк Bolgrad 0,25'])].rename_axis(None, axis=1)
btl_5_bg['Группа текущая'] = 'Коньяк Bolgrad 0,5'
btl_5_bg['Подтверждено'] = btl_5_bg['Коньяк Bolgrad 0,5'] - btl_5_bg['Удержание Коньяк Bolgrad 0,5']
btl_5_bg.rename(columns={'Коньяк Bolgrad 0,5':'Выдано','Удержание Коньяк Bolgrad 0,5':'Разница'},inplace=True)
# btl_5_bg.set_index(list(btl_5_bg.drop(['Выдано','Разница','Подтверждено'],axis=1).columns.values))


In [ ]:
#Join into one df previous separation
ressult_in_litre = btl_25_bg.append([btl_5_bg,btl_25_ar,btl_5_ar])
ressult_in_litre_for_join = ressult_in_litre.set_index(ressult_in_litre.drop(['Выдано','Разница','Подтверждено'],axis=1).columns.to_list())

In [ ]:
#Resulting df with difference
difference = ressult_in_non_litre.append(ressult_in_litre_for_join).reset_index()

In [ ]:
#Creating report with distos requesting and allowed issues
report = request.join(ressult_in_non_litre.append(ressult_in_litre_for_join).\
                groupby(by=['Дистрибьютор', 'Регион текущий','Группа текущая']).sum()['Разница'])
report.fillna(0,inplace=True)
report["Алколайн"] = report['Дистр'] - report['Разница']

In [ ]:
#Export
exp_report = report.reset_index()
exp_report[['Дистрибьютор', 'Регион текущий', 'Группа текущая', 'Дистр', 'Алколайн','Разница']].to_excel("отчет.xlsx",index=False)

In [ ]:
# Slice of promo df with points which have difference in confirming
difference_df = promo_df[promo_df["КодТорговойТочки"].isin(difference["КодТорговойТочки"])]
difference_df

In [ ]:
#Creating dummies and reformat columns for join
difference['Группа текущая 2'] = difference['Группа текущая']
difference.loc[difference['Группа текущая 2'].str.contains('0,'),'Группа текущая 2'] = difference['Группа текущая 2'].apply(lambda x: " ".join(x.split()[:-1]))
difference.set_index(difference.columns[~difference.columns.isin(['Выдано','Подтверждено','Разница','Группа текущая',])].to_list(),inplace=True)

In [ ]:
#Join difference and  points with sales
difference_mod = difference.join(difference_df.rename(columns={'Группа текущая':'Группа текущая 2'}).set_index(difference.index.names),how="left")
difference_mod.reset_index(inplace=True)
difference_mod

In [ ]:
#Adding explanation column, convering conditional promo values into real
difference_mod["Коментарий"] = np.nan
difference_mod.loc[difference_mod["Подарок"].isna(),"Коментарий"] = "Отгрузки на тт нет в бд"
difference_mod.loc[(difference_mod["Группа текущая"]=='Коньяк Bolgrad 0,5') & (~difference_mod["Подарок"].isna()),'Подарок'] = difference_mod['Подарок']/2
difference_mod.loc[(difference_mod["Группа текущая"]=='коньяк Армения 0,25') & (difference_mod["Коментарий"].isna()),'Подарок'] = difference_mod['Подарок']/3
difference_mod.loc[(difference_mod["Группа текущая"]=='коньяк Армения 0,5') & (difference_mod["Коментарий"].isna()),'Подарок'] = difference_mod['Подарок']/5
difference_mod['Подарок'].fillna(0,inplace=True)
difference_mod['Подарок'] = difference_mod['Подарок'].astype(int)

In [ ]:
# DF with difference in spliting promos with inconsistent values in checking/Adding reverse columns and joining with confirmed bottles
incos_difference_mod = difference_mod[difference_mod['Подарок']>=difference_mod['Выдано']]
incos = incos_difference_mod.drop(['Выдано', 'Подтверждено','Разница', 'Продажи вторичные, бут.', 'Продажи вторичные, л.','Количество SKU (2х)', 'Подарок', 'Дата', 'Коментарий', ],axis=1)
incos.loc[incos['Группа текущая']=='Коньяк Bolgrad 0,5','Группа текущая 2'] = 'Коньяк Bolgrad 0,25'
incos.loc[incos['Группа текущая']=='коньяк Армения 0,5','Группа текущая 2'] = 'коньяк Армения 0,25' 
incos = incos.set_index(['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки', 'Группа текущая 2']).join(confirmed_litre).reset_index()
incos.set_index(['Дистрибьютор', 'Регион текущий', 'Клиент', 'КодТорговойТочки', 'Группа текущая'],inplace=True)

In [ ]:
difference_mod[difference_mod['КодТорговойТочки']=='48-99905817']

In [ ]:
# Filling commentary column
for i in difference_mod.columns.to_list()[:-1]:
    difference_mod[i] = difference_mod[i].astype(str)
difference_mod.loc[(difference_mod['Коментарий'].isna()) & (~difference_mod['Группа текущая'].str.contains("оньяк")),'Коментарий'] = difference_mod['Дата'] + " продажа " + difference_mod["Продажи вторичные, бут."] + " бут " + " кол-во скю  " + difference_mod["Количество SKU (2х)"]  + " кол-во акц.бут  " + difference_mod['Подарок']
difference_mod.loc[(difference_mod['Коментарий'].isna()) & (difference_mod['Группа текущая'].str.contains("оньяк")),'Коментарий'] = difference_mod['Дата'] + " продажа " + difference_mod["Продажи вторичные, л."] + " л. " + " кол-во скю  " + difference_mod["Количество SKU (2х)"]   + " кол-во акц.бут  " + difference_mod['Подарок']
difference_mod.loc[(difference_mod['Группа текущая']=='Коньяк Болградский') & (difference_mod['Коментарий']!="Отгрузки на тт нет в бд"),'Коментарий'] = difference_mod['Дата'] + " продажа " + difference_mod["Продажи вторичные, бут."] + " бут " + " кол-во скю  " + difference_mod["Количество SKU (2х)"]  + " кол-во акц.бут  " + difference_mod['Подарок']
difference_mod.loc[(difference_mod['Группа текущая']=='Водка') & (difference_mod['Коментарий']!="Отгрузки на тт нет в бд"),'Коментарий'] = difference_mod['Дата'] + " продажа " + difference_mod["Продажи вторичные, л."] + " л. " + " кол-во скю  " + difference_mod["Количество SKU (2х)"]  + " кол-во акц.бут  " + difference_mod['Подарок']

In [ ]:
# Grouping droping
difference_mod.drop(['Продажи вторичные, бут.','Продажи вторичные, л.','Количество SKU (2х)','Подарок','Дата'],axis=1,inplace=True)
difference_mod['Коментарий'] = difference_mod.groupby(difference_mod.columns.to_list()[:-1])['Коментарий'].transform(lambda x : '\n'.join(x))
difference_mod.drop_duplicates(inplace=True)
difference_mod.loc[difference_mod["КодТорговойТочки"].isin(lpc['id']),"Коментарий"] = "ТТ самообслуживания"

In [ ]:
#Adding dummies and reformat for next joining 
for i in ['Выдано','Подтверждено','Разница']:
    difference_mod[i] = pd.to_numeric(difference_mod[i],downcast='signed')
difference_mod.drop('Группа текущая 2', axis=1,inplace=True)
difference_mod['Коментарий'] = difference_mod['Коментарий'].str.replace('\.0', '', regex=True)

In [ ]:
#Finish joininig
slave_go_home = difference_mod.set_index(['Дистрибьютор','Регион текущий','Клиент','КодТорговойТочки','Группа текущая']).join(incos,how="left").reset_index()
slave_go_home['Бут'] = slave_go_home['Бут'].apply(str)
slave_go_home.loc[~slave_go_home['Группа текущая 2'].isna(),'Коментарий'] = slave_go_home['Коментарий'] + "\n" "(Подтверждено " + slave_go_home["Группа текущая 2"] + " " + slave_go_home["Бут"] + ")"

In [ ]:
#Export
slave_go_home.drop(['Группа текущая 2','Бут'],axis=1,inplace=True)
slave_go_home.to_excel('расхождение.xlsx',index=False)

In [ ]:
#Check for no multiplying
difference.shape[0] == difference_mod.shape[0]

In [ ]:
request.reset_index().groupby(by='Дистрибьютор').sum().join(slave_go_home.groupby(by='Дистрибьютор')['Разница'].sum(),how='left')